In [3]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


def resolve_project_root() -> Path:
    """Resolve project root robustly from notebook cwd.

    Walks upward until a directory containing ``pyproject.toml`` is found.
    """
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


ROOT = resolve_project_root()

TASK_DIRS: dict[str, Path] = {
    "subtask_a": ROOT / "data" / "task_a",
    "subtask_b": ROOT / "data" / "task_b",
    "subtask_c": ROOT / "data" / "task_c",
}


def find_split_file(task_dir: Path, split_keyword: str) -> Path:
    """Find first parquet file containing split keyword."""
    candidates = [p for p in sorted(task_dir.glob("*.parquet")) if split_keyword in p.stem.lower()]
    if not candidates:
        raise FileNotFoundError(f"No '{split_keyword}' parquet in {task_dir}")
    return candidates[0]


FILES: dict[str, dict[str, Path]] = {
    task: {
        "train": find_split_file(task_dir, "train"),
        "validation": find_split_file(task_dir, "validation"),
        "test": find_split_file(task_dir, "test"),
    }
    for task, task_dir in TASK_DIRS.items()
}

ROOT

PosixPath('/Users/vietnq/HCMUS/ML/kaggle')

In [4]:
def load_df(path: Path) -> pd.DataFrame:
    """Load parquet into pandas dataframe."""
    return pd.read_parquet(path)


def basic_split_report(task: str, split: str, path: Path) -> dict[str, Any]:
    """Create a compact quality report for one split."""
    df = load_df(path)
    return {
        "task": task,
        "split": split,
        "rows": len(df),
        "columns": ", ".join(df.columns.astype(str)),
        "null_code": int(df["code"].isna().sum()) if "code" in df.columns else None,
        "null_label": int(df["label"].isna().sum()) if "label" in df.columns else None,
        "has_id": "id" in df.columns,
        "has_language": "language" in df.columns,
        "has_generator": "generator" in df.columns,
    }


report_rows: list[dict[str, Any]] = []
for task, splits in FILES.items():
    for split, path in splits.items():
        report_rows.append(basic_split_report(task, split, path))

report_df = pd.DataFrame(report_rows)
report_df

,task,split,rows,columns,null_code,null_label,has_id,has_language,has_generator
0,subtask_a,train,500000,"code, generator, label, language",0,0,False,True,True
1,subtask_a,validation,100000,"code, generator, label, language",0,0,False,True,True
2,subtask_a,test,1000,"code, generator, label, language",0,0,False,True,True
3,subtask_b,train,500000,"code, generator, label, language",0,0,False,True,True
4,subtask_b,validation,100000,"code, generator, label, language",0,0,False,True,True
5,subtask_b,test,1000,"code, generator, label, language",0,0,False,True,True
6,subtask_c,train,900000,"code, generator, label, language",0,0,False,True,True
7,subtask_c,validation,200000,"code, generator, label, language",0,0,False,True,True
8,subtask_c,test,1000,"code, generator, label, language",0,0,False,True,True


In [5]:
def snippet_preview(df: pd.DataFrame, n_per_label: int = 2, max_chars: int = 220) -> pd.DataFrame:
    """Return short code previews sampled by label."""
    if "label" not in df.columns or "code" not in df.columns:
        return pd.DataFrame()

    rows: list[dict[str, Any]] = []
    for label_value, group in df.groupby("label", dropna=False):
        sample = group.head(n_per_label)
        for _, row in sample.iterrows():
            code = str(row["code"])
            rows.append(
                {
                    "label": label_value,
                    "language": row.get("language", None),
                    "generator": row.get("generator", None),
                    "code_preview": code[:max_chars].replace("\n", " "),
                }
            )
    return pd.DataFrame(rows)


for task, splits in FILES.items():
    print(f"\n=== {task} sample code preview ===")
    train_df = load_df(splits["train"])
    display(snippet_preview(train_df, n_per_label=2).head(20))


=== subtask_a sample code preview ===


,label,language,generator,code_preview
0,0,Python,human,"(a, b, c, d) = [int(x) for x in input().split(..."
1,0,Python,human,T = int(input()) for t in range(T): \tcolor = ...
2,1,Python,Qwen/Qwen2.5-Coder-1.5B,valid version for the language; all others can...
3,1,Python,Qwen/Qwen2.5-Coder-7B-Instruct,python def min_cards_to_flip(s): vowels = ...



=== subtask_b sample code preview ===


,label,language,generator,code_preview
0,0,Python,Human,"def load(config, filepath, token): if conf..."
1,0,Python,Human,"n = int(input()) arr = list(map(int, input().s..."
2,1,Python,deepseek-ai/DeepSeek-V3-0324,import numpy as np from scipy.spatial.transfor...
3,1,C#,deepseek-ai/DeepSeek-V3-0324,using PiRhoSoft.Utilities; namespace PiRhoSof...
4,2,Java,Qwen/Qwen2.5-72B-Instruct,import java.util.ArrayList; import java.util.H...
5,2,C++,Qwen/Qwen2.5-Coder-1.5B-Instruct,#include<iostream> using namespace std; int ma...
6,3,Java,01-ai/Yi-Coder-1.5B-Chat,package com.example.layout.exceptions; public...
7,3,Python,01-ai/Yi-Coder-1.5B,'''Constants - `CONSTANT_NAME` specifies t...
8,4,C++,bigcode/starcoder2-3b,"using namespace std; int main() { int n,m; ci..."
9,4,Python,bigcode/starcoder2-15b,"\tt = int(input()) \tfor i in range(t): \t\ta,..."



=== subtask_c sample code preview ===


,label,language,generator,code_preview
0,0,PHP,Human,<?php use Fedeisas\LaravelJsRoutes\Commands\R...
1,0,Java,Human,// CodeForces // C. Alice and the Cake imp...
2,1,JavaScript,GPT-4o,"import React, { useState, useEffect } from 're..."
3,1,JavaScript,google/codegemma-7b-it,const math = require('mathjs'); /** * Calcul...
4,2,Python,01-ai/Yi-Coder-9B-Chat,"class Indexer: def __init__(self, token_li..."
5,2,C#,01-ai/Yi-Coder-1.5B-Chat,/// <summary> /// Handles the Click event of t...
6,3,Java,Qwen/Qwen2.5-Coder-1.5B-Instruct,"proto syntax = ""proto3""; message FetchResult ..."
7,3,Go,deepseek-ai/deepseek-coder-6.7b-instruct,/* This problem is pretty straightforward. Ac...


In [6]:
from transformers import AutoTokenizer


def length_stats(df: pd.DataFrame, tokenizer) -> pd.DataFrame:
    """Compute rough text and token length statistics."""
    text_series = df["code"].fillna("").astype(str)
    char_len = text_series.str.len()

    # Fast approximation: tokenize without truncation for first 2000 rows
    subset = text_series.head(2000).tolist()
    token_lens = [len(tokenizer.encode(t, add_special_tokens=True)) for t in subset]

    return pd.DataFrame(
        {
            "metric": ["rows", "chars_mean", "chars_p95", "tok_mean_2k", "tok_p95_2k"],
            "value": [
                len(df),
                round(float(char_len.mean()), 2),
                round(float(char_len.quantile(0.95)), 2),
                round(float(pd.Series(token_lens).mean()), 2),
                round(float(pd.Series(token_lens).quantile(0.95)), 2),
            ],
        }
    )


tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

for task, splits in FILES.items():
    print(f"\n=== {task} length stats (train) ===")
    train_df = load_df(splits["train"])
    display(length_stats(train_df, tokenizer))


=== subtask_a length stats (train) ===


Token indices sequence length is longer than the specified maximum sequence length for this model (1059 > 512). Running this sequence through the model will result in indexing errors


,metric,value
0,rows,500000.00
1,chars_mean,836.87
2,chars_p95,2751.00
3,tok_mean_2k,391.60
4,tok_p95_2k,1276.20



=== subtask_b length stats (train) ===


,metric,value
0,rows,500000.00
1,chars_mean,1409.68
2,chars_p95,4790.00
3,tok_mean_2k,667.89
4,tok_p95_2k,2441.60



=== subtask_c length stats (train) ===


,metric,value
0,rows,900000.00
1,chars_mean,1393.10
2,chars_p95,4023.00
3,tok_mean_2k,579.72
4,tok_p95_2k,1737.65
